# Exploración de `X_features.npy`

Este notebook inspecciona la matriz de características manuales generada desde los trials procesados.

Archivo principal:

`dataset/processed/representations/manual_deap_features/X_features.npy`

Objetivo:
- revisar shape,
- memoria en disco y RAM,
- nombres de características,
- metadata asociada,
- estadísticas básicas,
- posibles NaN/Inf,
- y ejemplos de acceso a un trial específico.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd


PROJECT_ROOT: Path = Path("/home/russell/ssd/code/Topicos_Ciencia_Datos/Visual_Analytic_DEAP")

FEATURE_DIR: Path = (
    PROJECT_ROOT
    / "dataset"
    / "processed"
    / "representations"
    / "manual_deap_features"
)

X_FEATURES_PATH: Path = FEATURE_DIR / "X_features.npy"
FEATURE_NAMES_PATH: Path = FEATURE_DIR / "feature_names.json"
TRIAL_METADATA_PATH: Path = FEATURE_DIR / "trial_metadata.csv"
FEATURE_CONFIG_PATH: Path = FEATURE_DIR / "feature_config.json"

In [2]:
paths: list[Path] = [
    X_FEATURES_PATH,
    FEATURE_NAMES_PATH,
    TRIAL_METADATA_PATH,
    FEATURE_CONFIG_PATH,
]

for path in paths:
    print(path.name, "exists:", path.exists())

X_features.npy exists: True
feature_names.json exists: True
trial_metadata.csv exists: True
feature_config.json exists: True


In [3]:
file_size_bytes: int = X_FEATURES_PATH.stat().st_size
file_size_mb: float = file_size_bytes / 1024**2

print(f"X_features.npy size: {file_size_bytes:,} bytes")
print(f"X_features.npy size: {file_size_mb:.4f} MB")

X_features.npy size: 3,233,440 bytes
X_features.npy size: 3.0836 MB


In [4]:
X_features: np.ndarray = np.load(X_FEATURES_PATH)

print("Shape:", X_features.shape)
print("Dtype:", X_features.dtype)
print("RAM:", X_features.nbytes / 1024**2, "MB")

Shape: (1279, 316)
Dtype: float64
RAM: 3.083526611328125 MB


In [5]:
with FEATURE_NAMES_PATH.open("r", encoding="utf-8") as file:
    feature_names: list[str] = json.load(file)

trial_metadata: pd.DataFrame = pd.read_csv(TRIAL_METADATA_PATH)

with FEATURE_CONFIG_PATH.open("r", encoding="utf-8") as file:
    feature_config: dict[str, Any] = json.load(file)

print("num feature names:", len(feature_names))
print("metadata shape:", trial_metadata.shape)

display(feature_config)
display(trial_metadata.head())

num feature names: 316
metadata shape: (1279, 17)


{'representation_type': 'manual_deap_features',
 'description': 'Feature vectors based on EEG spectral power/asymmetry and available physiological features inspired by DEAP Section 6.1/Table 5.',
 'num_trials': 1279,
 'num_features': 316,
 'eeg_features_expected': 216,
 'physiological_features': 100,
 'include_incomplete': False,
 'dtype': 'float64'}

,participant_id,trial,experiment_id,valence,arousal,dominance,liking,familiarity,event_source,sfreq,num_channels,num_samples,expected_samples,duration_sec,expected_duration_sec,is_complete,representation_input_npz
0,1,1,5,6.96,3.92,7.19,6.05,4.0,status,128.0,44,7680,7680,60.0,60.0,True,processed/representation_inputs/s01/trial_01_i...
1,1,2,18,7.23,7.15,6.94,8.01,4.0,status,128.0,44,7680,7680,60.0,60.0,True,processed/representation_inputs/s01/trial_02_i...
2,1,3,4,4.94,6.01,6.12,8.06,4.0,status,128.0,44,7680,7680,60.0,60.0,True,processed/representation_inputs/s01/trial_03_i...
3,1,4,24,7.04,7.09,8.01,8.22,4.0,status,128.0,44,7680,7680,60.0,60.0,True,processed/representation_inputs/s01/trial_04_i...
4,1,5,20,8.26,7.91,7.19,8.13,1.0,status,128.0,44,7680,7680,60.0,60.0,True,processed/representation_inputs/s01/trial_05_i...


In [6]:
print("X rows:", X_features.shape[0])
print("Metadata rows:", len(trial_metadata))
print("X cols:", X_features.shape[1])
print("Feature names:", len(feature_names))

assert X_features.shape[0] == len(trial_metadata)
assert X_features.shape[1] == len(feature_names)

print("OK: dimensiones consistentes.")

X rows: 1279
Metadata rows: 1279
X cols: 316
Feature names: 316
OK: dimensiones consistentes.


In [7]:
num_nan: int = int(np.isnan(X_features).sum())
num_inf: int = int(np.isinf(X_features).sum())

print("NaN values:", num_nan)
print("Inf values:", num_inf)

nan_columns: np.ndarray = np.where(np.isnan(X_features).any(axis=0))[0]
inf_columns: np.ndarray = np.where(np.isinf(X_features).any(axis=0))[0]

print("Columns with NaN:", len(nan_columns))
print("Columns with Inf:", len(inf_columns))

if len(nan_columns) > 0:
    display(pd.DataFrame({
        "feature_index": nan_columns,
        "feature_name": [feature_names[index] for index in nan_columns],
    }).head(30))

NaN values: 0
Inf values: 0
Columns with NaN: 0
Columns with Inf: 0


In [8]:
global_stats: dict[str, float] = {
    "min": float(np.nanmin(X_features)),
    "max": float(np.nanmax(X_features)),
    "mean": float(np.nanmean(X_features)),
    "std": float(np.nanstd(X_features)),
}

display(global_stats)

{'min': -240639.19581087248,
 'max': 285817.6819937297,
 'mean': 403.95938212676884,
 'std': 5963.493559797834}

In [9]:
features_df: pd.DataFrame = pd.DataFrame(
    X_features,
    columns=feature_names,
)

display(features_df.head())

,EEG__Fp1__log_power__theta,EEG__Fp1__log_power__slow_alpha,EEG__Fp1__log_power__alpha,EEG__Fp1__log_power__beta,EEG__Fp1__log_power__gamma,EEG__AF3__log_power__theta,EEG__AF3__log_power__slow_alpha,EEG__AF3__log_power__alpha,EEG__AF3__log_power__beta,EEG__AF3__log_power__gamma,...,EMG_EXG7__rms,EMG_EXG7__mean_abs_derivative,EMG_EXG7__log_power_4_40Hz,EMG_EXG8__mean,EMG_EXG8__std,EMG_EXG8__min,EMG_EXG8__max,EMG_EXG8__rms,EMG_EXG8__mean_abs_derivative,EMG_EXG8__log_power_4_40Hz
0,-25.805807,-26.770895,-26.115036,-25.578528,-27.007158,-25.972414,-26.891903,-26.260490,-25.708968,-27.085470,...,0.002695,0.000028,-20.250354,-0.004549,0.000070,-0.004736,-0.004220,0.004550,0.000028,-20.752229
1,-26.183843,-27.108654,-26.631346,-25.988537,-27.194478,-26.176707,-27.063995,-26.613709,-25.873786,-27.187446,...,0.000882,0.000028,-20.213712,-0.004024,0.000083,-0.004192,-0.003645,0.004025,0.000028,-20.721668
2,-25.508048,-26.729876,-26.013647,-25.405061,-26.885256,-25.669244,-26.838828,-26.194090,-25.579899,-27.063993,...,0.000935,0.000027,-20.158342,-0.003881,0.000107,-0.004194,-0.003513,0.003882,0.000026,-20.673751
3,-25.691279,-26.752305,-26.039139,-25.369842,-26.845591,-25.910619,-26.889385,-26.238420,-25.547560,-27.052857,...,0.000947,0.000027,-20.225205,-0.003762,0.000055,-0.003918,-0.003469,0.003762,0.000026,-20.731585
4,-25.813066,-26.791172,-26.126061,-25.473956,-26.856670,-25.965626,-26.889987,-26.255147,-25.495663,-26.978376,...,0.000703,0.000026,-20.253298,-0.003875,0.000141,-0.004208,-0.003477,0.003878,0.000026,-20.756720


In [10]:
eeg_feature_names: list[str] = [
    name for name in feature_names
    if name.startswith("EEG__")
]

print("EEG features:", len(eeg_feature_names))
display(pd.DataFrame({"feature_name": eeg_feature_names}).head(30))
display(pd.DataFrame({"feature_name": eeg_feature_names}).tail(30))

EEG features: 216


,feature_name
0,EEG__Fp1__log_power__theta
1,EEG__Fp1__log_power__slow_alpha
2,EEG__Fp1__log_power__alpha
3,EEG__Fp1__log_power__beta
4,EEG__Fp1__log_power__gamma
5,EEG__AF3__log_power__theta
6,EEG__AF3__log_power__slow_alpha
7,EEG__AF3__log_power__alpha
8,EEG__AF3__log_power__beta
9,EEG__AF3__log_power__gamma


,feature_name
186,EEG__asymmetry__C3_minus_C4__beta
187,EEG__asymmetry__C3_minus_C4__gamma
188,EEG__asymmetry__T7_minus_T8__theta
189,EEG__asymmetry__T7_minus_T8__alpha
190,EEG__asymmetry__T7_minus_T8__beta
191,EEG__asymmetry__T7_minus_T8__gamma
192,EEG__asymmetry__CP5_minus_CP6__theta
193,EEG__asymmetry__CP5_minus_CP6__alpha
194,EEG__asymmetry__CP5_minus_CP6__beta
195,EEG__asymmetry__CP5_minus_CP6__gamma


In [11]:
physio_feature_names: list[str] = [
    name for name in feature_names
    if not name.startswith("EEG__")
]

print("Physiological features:", len(physio_feature_names))
display(pd.DataFrame({"feature_name": physio_feature_names}).head(50))

Physiological features: 100


,feature_name
0,GSR__mean
1,GSR__std
2,GSR__min
3,GSR__max
4,GSR__rms
5,GSR__mean_abs_derivative
6,GSR__mean_derivative
7,GSR__mean_negative_derivative
8,GSR__proportion_negative_derivative
9,GSR__num_peaks


In [12]:
participant_id: int = 1
trial: int = 2

mask: pd.Series = (
    (trial_metadata["participant_id"] == participant_id)
    & (trial_metadata["trial"] == trial)
)

trial_indices: list[int] = trial_metadata.index[mask].tolist()

print("Matching indices:", trial_indices)

trial_index: int = trial_indices[0]
trial_vector: np.ndarray = X_features[trial_index]

print("Trial index:", trial_index)
print("Vector shape:", trial_vector.shape)
print("Experiment ID:", trial_metadata.loc[trial_index, "experiment_id"])

Matching indices: [1]
Trial index: 1
Vector shape: (316,)
Experiment ID: 18


In [13]:
top_k: int = 20

top_indices: np.ndarray = np.argsort(np.abs(trial_vector))[::-1][:top_k]

top_features_df: pd.DataFrame = pd.DataFrame(
    {
        "feature_index": top_indices,
        "feature_name": [feature_names[index] for index in top_indices],
        "value": trial_vector[top_indices],
        "abs_value": np.abs(trial_vector[top_indices]),
    }
)

display(top_features_df)

,feature_index,feature_name,value,abs_value
0,219,GSR__max,3602.462530,3602.462530
1,220,GSR__rms,3447.826140,3447.826140
2,216,GSR__mean,3446.873273,3446.873273
3,218,GSR__min,3320.676113,3320.676113
4,225,GSR__num_peaks,2481.000000,2481.000000
5,226,GSR__num_local_minima,2480.000000,2480.000000
6,217,GSR__std,81.053852,81.053852
7,251,Plet__heart_rate_per_min,60.000000,60.000000
8,250,Plet__num_peaks,60.000000,60.000000
9,235,Resp__breathing_rate_per_min,46.000000,46.000000


## Conclusión

La matriz `X_features.npy` contiene una fila por trial completo y una columna por característica fisiológica/EEG.

En esta versión:

- se generaron 1279 trials completos,
- cada trial tiene 316 características,
- las primeras características corresponden al bloque EEG,
- las restantes corresponden a señales fisiológicas disponibles,
- y esta matriz será la entrada para normalización, PCA, UMAP o t-SNE.